In [ ]:
import os
import shutil

# 0. Bulletproof Setup: Check for repo and enforce working directory
repo_path = "/kaggle/working/rl_sf"
if not os.path.exists(repo_path):
    print("--> Repository missing. Cloning now...")
    !git clone -b optimization https://github.com/flaviogeuforbio/rl-with-sf-for-mujoco {repo_path}

# Change directory explicitly to where the script lives
%cd {repo_path}

In [ ]:
import torch
print("torch:", torch.version)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
!python -m pip install mujoco

In [ ]:
%ls

!ls -l /kaggle/input/

In [ ]:
!ls -R /kaggle/input/datasets

In [ ]:
import os
import shutil

# --- PHASE 1 SMOKE TEST: INITIAL RUN ---

STEPS_PER_PHASE = "4000"  
SAVE_FREQ = "1000"
RUN_STEPS_LIMIT = "2000"
GAMMA_VAL = "0.99"
LAMBDA_Q = "1.0" 
LAMBDA_VEC = "1.0" 
RESUME_DIR = "/kaggle/input/datasets/adrianoarceri/checkpoint" # change the name of the last folder according to the name you give to the dataset...

# Removed stepsxphase to keep the directory name constant across resumed chunks
RUN_NAME_SEQ = f"3DFeatures_Walker_transfer_gamma_{GAMMA_VAL.replace('.', '_')}_lq_{LAMBDA_Q.replace('.', '_')}_lvec_{LAMBDA_VEC.replace('.', '_')}"

kaggle_output_folder = "/kaggle/working/transfer_learning_long_run"
os.makedirs(kaggle_output_folder, exist_ok=True)

local_path_seq = f"/kaggle/working/rl_sf/artifacts/walker/{RUN_NAME_SEQ}"

seed=1 
print(f"\n--- EXECUTING SEED {seed} ---")

print("-> Running Sequential Training...")
!python -u train_cheetah_walker.py --steps_per_phase {STEPS_PER_PHASE} --save_freq {SAVE_FREQ} --baseline --run_name {RUN_NAME_SEQ} --gamma {GAMMA_VAL} --lambda_q {LAMBDA_Q} --lambda_vec {LAMBDA_VEC} --seed {seed} --resume_dir {RESUME_DIR} --run_steps_limit {RUN_STEPS_LIMIT}

if os.path.exists(local_path_seq):
    final_dest_seq = os.path.join(kaggle_output_folder, RUN_NAME_SEQ)
    shutil.copytree(local_path_seq, final_dest_seq, dirs_exist_ok=True)
    print(f"--> Sequential data saved in: {final_dest_seq}")

print("\n--> Zipping results for download...")
%cd /kaggle/working/
!zip -r transfer_learning_long_run.zip transfer_learning_long_run/
%cd /kaggle/working/rl_sf